# Ejercicio Práctico — Semana 3: Emociones en texto: NLP, clasificación y procesamiento distribuido

**Asignatura:** Inteligencia Artificial: Data Mining II — Maestría en Ciencias de Datos y Máquinas de Aprendizaje, mención IA (UIDE)
**Docente:** Miguel Angel Rosero Aguas
**Cubre:** Sesión 1 — Big Data con PySpark: Online Retail II  /  Sesión 2 — NLP, spaCy, LDA y Spark MLlib
**Grupo N.º:** ___

| # | Integrante | Rol en el equipo |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |

## Índice

1. [Carga, limpieza y exploración del corpus](#fase-1)
2. [Análisis lingüístico con spaCy: POS y NER](#fase-2)
3. [Modelado de temas (LDA)](#fase-3)
4. [Clasificación clásica: Naive Bayes y SVM](#fase-4)
5. [Red neuronal recurrente (RNN)](#fase-5)
6. [Experimento: número de clases](#fase-6)
7. [Big Data: PySpark, MLlib y MapReduce](#fase-7)
8. [Desafío y evaluación de alternativas](#fase-8)
9. [Tendencias: BERTopic, LLM en el pipeline y Parquet](#fase-9)
10. [Fase final: aplicación al entorno profesional](#fase-final)

## Estructura del ejercicio

Este notebook va con la guía **Ejercicio_Practico_Semana3_Guia.docx**: el código y los gráficos se hacen aquí; en la guía responden por escrito
la **pregunta de cada fase** (la misma que aparece abajo de cada título) y la fase final individual.

| Bloque | Fases | Qué entregan |
|---|---|---|
| Datos y exploración | 1–3 | código + conclusión en el notebook; respuesta en la guía |
| Modelado y análisis | 4–7 | código + conclusión en el notebook; respuesta en la guía |
| Desafío y tendencias | 8–9 | evidencia en el notebook; respuesta en la guía |
| Aplicación profesional | final | una respuesta por integrante (notebook y guía) |

**Entrega en Canvas:** este notebook ejecutado y la guía completada (Word o PDF).

## Contexto

El área de experiencia del cliente de una empresa recibe miles de mensajes al día y quiere detectar automáticamente la emoción de cada uno para priorizar la atención. Antes de invertir, les pide un estudio con un corpus público de tweets etiquetados con 13 emociones: qué dice el texto (análisis lingüístico y temas), qué tan bien se puede clasificar la emoción según cuántas emociones se distingan, y si hace falta una plataforma distribuida (Spark/Hadoop) para su volumen.

**Dataset:** Emotion Detection from Text (Kaggle, pashupatigupta): https://www.kaggle.com/datasets/pashupatigupta/emotion-detection-from-text (tweet_emotions.csv: 40 000 tweets, columnas tweet_id, sentiment, content)

## Desafío

1) Recomendar la **granularidad de emociones** (número de clases) que el área puede usar con confianza, con evidencia del experimento de clases y una línea base. 2) Recomendar el **modelo** (Naive Bayes, SVM o RNN) con su costo y su desempeño. 3) Estimar si su volumen justifica **Spark** y cómo se desplegaría (HDFS/S3, clúster en AWS), con evidencia del pipeline en PySpark. 4) Declarar una limitación seria del dataset y una implicación ética.

> **Formato de entrega:** un único `.ipynb` ejecutado. Código limpio y comentado. Antes de cada gráfico,
> una línea que diga qué se analiza; después, la conclusión de negocio. Nombre del archivo:
> `grupoN_Semana3_Ejercicio.ipynb`.

In [ ]:
# ===== Entorno: el mismo notebook corre en Google Colab y en local =====
import importlib, importlib.util, os, shutil, subprocess, sys, urllib.request   # utilidades: instalar paquetes, rutas y descargas
from pathlib import Path                            # rutas iguales en Windows, Linux y Colab

EN_COLAB = "google.colab" in sys.modules           # True si estamos dentro de Colab
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Python", sys.version.split()[0])   # dónde corre y qué Python

# Instalamos solo los paquetes que falten. El nombre en pip no siempre es el del import:
# ej.: pip install scikit-learn -> import sklearn; nx-parallel -> nx_parallel
IMPORT_DE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",   # nombre en pip -> nombre del import
             "delta-spark": "delta", "python-calamine": "python_calamine",   # instaladores y lectores de Excel
             "python-louvain": "community", "imbalanced-learn": "imblearn"}   # comunidades y desbalance de clases
for pip_nombre in ['pyspark', 'spacy', 'kagglehub', 'bertopic', 'sentence-transformers']:   # paquetes que necesita esta práctica
    base = pip_nombre.split("==")[0].split(">=")[0]                    # ej.: "networkx>=3.7" -> "networkx"
    modulo = IMPORT_DE.get(base, base.replace("-", "_"))   # nombre del import. Ej.: nx-parallel -> nx_parallel
    if importlib.util.find_spec(modulo) is None:   # solo si NO está instalado (Colab ya trae muchos)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_nombre])   # Colab y venv con pip
        if r.returncode != 0 and shutil.which("uv"):                   # venv creado con uv (no trae pip)
            r = subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, pip_nombre])   # instala con uv en este mismo Python
        if r.returncode != 0:   # ni pip ni uv pudieron instalarlo
            raise RuntimeError(f"Instala '{pip_nombre}' en este entorno (pip/uv) y vuelve a ejecutar")   # error explícito con la acción a seguir
        importlib.invalidate_caches()                                  # para que el import vea el paquete recién instalado

# Spark corre sobre la JVM: en Colab instalamos Java 17; en local debe existir JAVA_HOME
if EN_COLAB and not os.environ.get("JAVA_HOME"):   # solo en Colab y si Java aún no está
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)   # instala Java 17 sin mensajes (tarda ~1 min)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"   # dónde lo deja apt en Colab
if not EN_COLAB and not os.environ.get("JAVA_HOME") and shutil.which("java") is None:   # local sin JAVA_HOME ni java en el PATH
    # Buscamos un JDK 17 en rutas típicas. Ej.: ~/opt/jdk17 (WSL), /usr/lib/jvm/java-17-openjdk-amd64 (Ubuntu)
    import glob   # glob busca rutas con comodines. Ej.: /usr/lib/jvm/*17*
    for jdk in [str(Path.home() / "opt" / "jdk17"), *glob.glob("/usr/lib/jvm/*17*"),   # rutas candidatas de un JDK 17
                *glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*")]:   # Windows con Temurin 17
        if Path(jdk, "bin").exists():   # es un JDK si tiene la carpeta bin
            os.environ["JAVA_HOME"] = jdk                                    # Spark lo lee al arrancar
            os.environ["PATH"] = str(Path(jdk, "bin")) + os.pathsep + os.environ["PATH"]   # el comando java queda disponible
            break   # se usa el primero que exista
if not EN_COLAB and not os.environ.get("JAVA_HOME") and shutil.which("java") is None:   # no se encontró ningún JDK
    raise RuntimeError("Spark necesita Java 17: instálalo (por ejemplo, Eclipse Temurin 17), define JAVA_HOME "   # sin Java, Spark no arranca
                       "y reinicia el kernel")   # acción para corregirlo
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(java en el PATH)"))   # confirma qué Java usará Spark
# Los workers de Spark deben usar el MISMO Python que este notebook (si no: PYTHON_VERSION_MISMATCH)
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable   # mismo Python en driver y workers
# Carpeta de datos relativa: en Colab es /content/data, en local ./data
DATA = Path("data"); DATA.mkdir(exist_ok=True)   # crea la carpeta si no existe

def descargar(url, nombre):   # Ej.: descargar(URL, "telco.csv") -> data/telco.csv
    # Descarga una sola vez; si el archivo ya existe, no vuelve a bajarlo
    destino = DATA / nombre   # ruta local del archivo. Ej.: data/telco.csv
    if not destino.exists():   # descarga solo la primera vez
        print("Descargando", nombre, "...")   # aviso de progreso
        urllib.request.urlretrieve(url, destino)   # descarga el archivo en data/
    return destino   # devuelve la ruta para usarla después

ARCHIVOS = [   # lista de (url, nombre) que usa esta práctica
    
]
for url, nombre in ARCHIVOS:   # descarga cada archivo
    print(nombre, "->", round(descargar(url, nombre).stat().st_size / 1e6, 2), "MB")   # y muestra su tamaño en MB

In [ ]:
# ===== Estilo de gráficos (principios de Tufte): alta proporción dato-tinta, un solo color de acento =====
import matplotlib.pyplot as plt                     # gráficos
from cycler import cycler                           # orden de colores por defecto
TINTA, ACENTO, GRIS = "#14323F", "#6E5BD6", "#B7C3C9"     # tokens del módulo; ACENTO = color de esta semana
plt.rcParams.update({   # estilo por defecto de todos los gráficos
    "axes.spines.top": False, "axes.spines.right": False,    # sin marco: solo los ejes necesarios
    "axes.grid": False, "axes.edgecolor": GRIS,              # sin rejilla (las etiquetas dicen el valor)
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlecolor": TINTA,   # títulos a la izquierda y en negrita
    "axes.prop_cycle": cycler(color=[TINTA, ACENTO, GRIS, "#5B6B73"]),
    "font.size": 12, "legend.frameon": False, "figure.dpi": 110,   # letra legible, leyenda sin caja, nitidez
})
# Regla práctica: pinten con ACENTO solo el dato que quieren que el lector mire; el resto en GRIS.

<a id="fase-1"></a>
## Fase 1: Carga, limpieza y exploración del corpus

Paso 1 del enunciado. Carguen el CSV, revisen duplicados, longitud de los textos y la distribución de las 13 emociones (¿qué tan desbalanceada está? ¿qué significa la etiqueta `empty`?). Limpien menciones, URLs y entidades HTML, y **documenten** cada decisión de limpieza.

> **Pregunta de la guía:** Duplicados, longitud media y distribución de las 13 emociones. ¿Qué significa «empty» y qué hicieron con ella?

**Qué analizamos:** _completar antes del código_

In [ ]:
# Modelo de spaCy en inglés: se descarga solo la primera vez (Colab o local)
import spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    spacy.cli.download('en_core_web_sm')

import numpy as np, pandas as pd, sklearn, kagglehub
SEMILLA = 42
ruta = kagglehub.dataset_download('pashupatigupta/emotion-detection-from-text')
df = pd.read_csv(f'{ruta}/tweet_emotions.csv')
print(df.shape, '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)
# completar: duplicados, longitud de textos, distribución de 'sentiment', limpieza (@menciones, URLs, &amp;)


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-2"></a>
## Fase 2: Análisis lingüístico con spaCy: POS y NER

Sobre una muestra estratificada: tokenización, lematización, **etiquetado gramatical (POS)**, **reconocimiento de entidades (NER)** y **análisis de dependencias** (al menos 3 tweets, con `displacy`). Comparen la proporción de adjetivos, verbos e interjecciones entre emociones y las entidades más frecuentes. ¿Aporta información útil para clasificar?

> **Pregunta de la guía:** ¿Qué categorías gramaticales, entidades y dependencias distinguen a las emociones? ¿Aportan para clasificar?

**Qué analizamos:** _completar antes del código_

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser'])  # solo lo necesario: tagger + NER
# Procesar en lotes con nlp.pipe (mucho más rápido que llamar nlp() fila por fila)
# Dependencias: modelo completo (con parser) en 3 tweets -> token.dep_, token.head; spacy.displacy.render(doc, style='dep')
# muestra = df.groupby('sentiment').sample(200, random_state=SEMILLA)
# for doc in nlp.pipe(muestra['texto_limpio'], batch_size=500): ... token.pos_, token.lemma_, doc.ents


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-3"></a>
## Fase 3: Modelado de temas (LDA)

Apliquen LDA (scikit-learn) sobre la bolsa de palabras. Elijan el número de temas con un criterio explícito (perplejidad o coherencia), interpreten los temas con sus palabras principales y crúcenlos con las emociones.

> **Pregunta de la guía:** Número de temas elegido (criterio), palabras clave de 3 temas y su cruce con emociones.

**Qué analizamos:** _completar antes del código_

In [ ]:
# Completar


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-4"></a>
## Fase 4: Clasificación clásica: Naive Bayes y SVM

Paso 2. Partición **estratificada** entrenamiento/validación/prueba con semilla. Dos pipelines de scikit-learn: TF-IDF + Naive Bayes multinomial y TF-IDF + SVM lineal. Ajusten en validación y reporten F1 macro y matriz de confusión (13 clases). La prueba se usa una sola vez.

> **Pregunta de la guía:** F1 macro y matriz de confusión de NB y SVM. ¿Qué emociones confunden más?

**Qué analizamos:** _completar antes del código_

In [ ]:
# Completar


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-5"></a>
## Fase 5: Red neuronal recurrente (RNN)

Paso 2. Modelo en Keras: `TextVectorization` → `Embedding` → LSTM o GRU (bidireccional opcional) → softmax. Usen la misma partición, early stopping sobre validación y pesos de clase o muestreo para el desbalance. Comparen costo de entrenamiento y desempeño frente a NB y SVM.

> **Pregunta de la guía:** F1 macro, tiempo de entrenamiento y estrategia de desbalance de la RNN frente a NB y SVM.

**Qué analizamos:** _completar antes del código_

In [ ]:
# Completar


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-6"></a>
## Fase 6: Experimento: número de clases

Pasos 3–4. Repitan los tres modelos con **3, 5, 7, 9 y 11** emociones seleccionadas al azar (fijen y reporten la semilla; idealmente promedien 3 semillas). Grafiquen F1 macro frente al número de clases para cada modelo, junto a la línea base de azar (1/k). Reporten **accuracy y F1 macro**: si divergen mucho para un mismo subconjunto, revisen qué clases salieron sorteadas y su tamaño. Expliquen la tendencia.

> **Pregunta de la guía:** Gráfica F1 macro vs. número de clases con la línea de azar. ¿Qué granularidad recomiendan?

**Qué analizamos:** _completar antes del código_

In [ ]:
# Completar


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-7"></a>
## Fase 7: Big Data: PySpark, MLlib y MapReduce

Repliquen la clasificación con un **Pipeline de Spark MLlib** (tokenización, stopwords, TF-IDF, Naive Bayes o regresión logística) y un **conteo de palabras estilo MapReduce** con RDD. Muestren el plan con `explain()`, una consulta equivalente en **Spark SQL** (`createOrReplaceTempView`) y una **función de ventana** (top-5 palabras por emoción con `row_number()`). Relacionen cada paso con Hadoop (HDFS para almacenamiento, MapReduce frente a Spark en memoria) y describan cómo se desplegaría en un clúster en AWS (EMR, datos en S3). **Spark 4:** expliquen qué cambia con Spark Connect (el notebook como cliente de un clúster remoto, `SparkSession.builder.remote(...)`) y por qué en ese modo no hay RDD.

> **Pregunta de la guía:** Plan de explain(), consulta Spark SQL con función de ventana, conteo MapReduce y despliegue en EMR con S3.

**Qué analizamos:** _completar antes del código_

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import NaiveBayes
spark = (SparkSession.builder.appName('emociones').master('local[*]')
         .config('spark.driver.memory', '4g').getOrCreate())  # 4 GB: el valor por defecto (1 GB) se queda corto con TF-IDF
sdf = spark.createDataFrame(df[['texto_limpio', 'sentiment']])  # en producción: spark.read.csv('s3a://...') o HDFS
# completar: Pipeline MLlib (tokenizar → stopwords → TF → IDF → indexar etiqueta → modelo), split, evaluación
# MapReduce: conteo de palabras con RDD → sdf.rdd.flatMap(...).map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-8"></a>
## Fase 8: Desafío y evaluación de alternativas

Respondan el desafío con evidencia. Completen una tabla de **evaluación de alternativas**: técnica (spaCy, LDA, NB, SVM, RNN, scikit-learn vs. Spark) · ventajas · limitaciones · cuándo la usarían en un escenario profesional o de investigación. Citen lecturas del Tema 3 y documentación oficial.

> **Pregunta de la guía:** Tabla de alternativas y respuesta al desafío (granularidad, modelo y decisión sobre Spark).

**Qué analizamos:** _completar antes del código_

In [ ]:
alternativas = pd.DataFrame(columns=['tecnica', 'ventajas', 'limitaciones', 'aplicabilidad', 'fuente'])


**Conclusión de negocio:** _completar después del resultado_

La **fase de tendencias** usa un modelo de lenguaje. Esta celda lo prepara (Ollama en local, Hugging Face en Colab):

In [ ]:
# ===== Modelo de lenguaje (LLM): Ollama en local o un modelo pequeño de Hugging Face en Colab =====
import json, os, urllib.request                       # json: armar/leer mensajes; urllib: peticiones HTTP a la API de Ollama

def _hosts_ollama():   # lista de direcciones donde buscar Ollama
    """Lugares donde puede estar Ollama. Ej.: en Windows es localhost; desde WSL es la IP del Windows anfitrión."""
    hosts = [os.environ.get("OLLAMA_HOST", ""), "http://localhost:11434"]   # OLLAMA_HOST permite forzar otro equipo
    try:                                              # en WSL, el 'nameserver' de resolv.conf es el Windows anfitrión
        ip = [l.split()[1] for l in open("/etc/resolv.conf") if l.startswith("nameserver")][0]   # IP del Windows anfitrión vista desde WSL
        hosts.append(f"http://{ip}:11434")            # ej.: http://172.29.80.1:11434
    except Exception:                                 # en Colab o Windows no hay WSL: se ignora
        pass   # sin host adicional
    return [h if h.startswith("http") else f"http://{h}" for h in hosts if h]   # normaliza: siempre con http://

OLLAMA, MODELO_OLLAMA = None, os.environ.get("MODELO_OLLAMA")  # ej.: MODELO_OLLAMA="llama3.2:3b" para elegir modelo
for h in _hosts_ollama():   # prueba cada dirección
    try:                                              # /api/tags lista los modelos instalados; timeout corto
        modelos = [m["name"] for m in json.load(urllib.request.urlopen(f"{h}/api/tags", timeout=2))["models"]]
        chat = [m for m in modelos if "embed" not in m]   # descartamos modelos de embeddings (no generan texto)
        if chat:   # hay al menos un modelo de chat
            OLLAMA, MODELO_OLLAMA = h, MODELO_OLLAMA or chat[0]   # ej.: ("http://localhost:11434", "qwen3.6:35b-a3b")
            break   # usamos el primero que responde
    except Exception:                                 # no responde: probamos el siguiente host
        continue   # siguiente dirección

try:                                                  # torch solo hace falta sin Ollama (Colab lo trae instalado)
    import torch                                      # para saber si hay GPU (Colab: Entorno > T4 GPU)
    GPU = torch.cuda.is_available()                   # True si hay GPU disponible
except ImportError:                                   # local sin torch: basta con Ollama
    GPU = False                                       # sin torch asumimos CPU
# Con GPU usamos 1.5B (mejor calidad, ~3 GB); sin GPU, 0.5B (~1 GB) para acotar el tiempo de inferencia en CPU
MODELO_HF = "Qwen/Qwen2.5-1.5B-Instruct" if GPU else "Qwen/Qwen2.5-0.5B-Instruct"
# Nota WSL: para usar el Ollama de Windows desde WSL, inicien Ollama en Windows con OLLAMA_HOST=0.0.0.0
_generador = None                                     # se carga solo la primera vez que se necesita
print("LLM:", f"Ollama {MODELO_OLLAMA} en {OLLAMA}" if OLLAMA else f"Hugging Face {MODELO_HF} (se descarga la 1.ª vez)")

def preguntar_llm(prompt, max_tokens=200):   # misma función en local (Ollama) y en Colab (Hugging Face)
    """Envía un prompt y devuelve el texto. Ej.: preguntar_llm("Resume en una frase: ...") -> "El texto trata de ..." """
    global _generador   # reutiliza el modelo ya cargado
    if OLLAMA:                                        # camino 1: servidor Ollama (rápido, privado, sin internet)
        cuerpo = json.dumps({"model": MODELO_OLLAMA, "prompt": prompt, "stream": False, "think": False,
                             "options": {"temperature": 0, "num_predict": max_tokens}}).encode()  # temperature 0 = repetible
        req = urllib.request.Request(f"{OLLAMA}/api/generate", cuerpo, {"Content-Type": "application/json"})   # petición POST a Ollama
        return json.load(urllib.request.urlopen(req, timeout=600))["response"].strip()   # texto de la respuesta
    if _generador is None:                            # camino 2: transformers (Colab ya lo trae instalado)
        from transformers import pipeline   # carga perezosa: solo si no hay Ollama
        _generador = pipeline("text-generation", model=MODELO_HF,
                              device=0 if GPU else -1)   # 0 = primera GPU, -1 = CPU
    salida = _generador([{"role": "user", "content": prompt}], max_new_tokens=max_tokens, do_sample=False)
    return salida[0]["generated_text"][-1]["content"].strip()   # el último mensaje es la respuesta del modelo

<a id="fase-9"></a>
## Fase 9: Tendencias: BERTopic, LLM en el pipeline y Parquet

a) **BERTopic** sobre los mismos tweets de la fase 3: comparen con LDA el número de temas, sus palabras clave, el % de tweets sin tema (-1) y el cruce con emociones. b) **LLM zero-shot:** clasifiquen 100 tweets de prueba con `preguntar_llm()` (sin entrenar) y compárenlo con su mejor modelo de la fase 4 en F1 macro, tiempo por tweet y respuestas fuera de la lista. c) **Embeddings:** entrenen el mismo clasificador lineal sobre embeddings (`all-MiniLM-L6-v2`) y compárenlo con TF-IDF en macro-F1. d) **Drift:** ordenen por `tweet_id`, entrenen con el tramo más antiguo y midan el PSI de las predicciones en el tramo más reciente. e) Guarden la tabla de resultados en **Parquet**, léanla con Spark y comparen su tamaño con el CSV equivalente. Cierren: ¿en qué escenario se justifica un LLM o un modelo de embeddings frente a TF-IDF + SVM? (costo, privacidad LOPDP, latencia). Citen la lectura de tendencia de la semana (*¿LLM o embeddings? El dilema del costo en la minería de textos*), BERTopic (Grootendorst, 2022) y la documentación de spacy-llm.

> **Pregunta de la guía:** BERTopic vs. LDA; LLM zero-shot y embeddings vs. su mejor modelo (F1 macro, tiempo, costo); PSI de drift; tamaño CSV vs. Parquet.

**Qué analizamos:** _completar antes del código_

In [ ]:
# --- a) BERTopic frente a LDA (fase 3) sobre los mismos textos ---
from bertopic import BERTopic                           # temas con embeddings + clustering + c-TF-IDF
from sentence_transformers import SentenceTransformer   # convierte cada tweet en un embedding semántico
muestra = df.sample(5000, random_state=SEMILLA)         # 5 000 tweets: corre en minutos en CPU, segundos en GPU
modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")    # 384 dimensiones; ~90 MB, se descarga la 1.ª vez
temas = BERTopic(embedding_model=modelo_emb, min_topic_size=30, language="english")   # min_topic_size: tamaño mínimo
asignados, _ = temas.fit_transform(muestra["texto_limpio"].tolist())                   # tema de cada tweet (-1 = ruido)
temas.get_topic_info().head(10)                          # ej.: tema 0 -> "birthday, happy, wish, ..."
# completar: comparar con LDA (número de temas, palabras clave, % de ruido) y cruzar temas con emociones

# --- b) LLM zero-shot frente a su mejor modelo clásico (fase 4), en los MISMOS 100 tweets de prueba ---
# etiquetas = sorted(df["sentiment"].unique())
# pred_llm = [preguntar_llm(f"Clasifica la emoción del tweet con UNA palabra de esta lista {etiquetas}. "
#                           f"Tweet: {t}\nEmoción:", max_tokens=5).strip().lower() for t in X_prueba[:100]]
# completar: F1 macro del LLM vs. SVM/NB en esos 100; tiempo por tweet; respuestas fuera de la lista

# --- c) Guardar resultados en Parquet y leerlos con Spark (formato columnar del ecosistema Big Data) ---
# resultados.to_parquet("data/resultados_emociones.parquet")              # pandas -> Parquet
# spark.read.parquet("data/resultados_emociones.parquet").groupBy("modelo").avg("f1_macro").show()


**Conclusión de negocio:** _completar después del resultado_

<a id="fase-final"></a>
## Fase final: aplicación al entorno profesional (individual)

¿Qué textos no estructurados se generan en tu entorno laboral (correos, reclamos, chats, informes, redes sociales) en Ecuador o tu país, y en qué idioma o variante? Describe qué información extraerías (emoción, temas, entidades), qué técnica de esta semana usarías, si necesitarías procesamiento distribuido, y una implicación ética de analizar esos textos.

Cada integrante responde **por separado** (150–250 palabras). Esta sección se califica de forma individual.

### Integrante 1: _nombre_

_Respuesta individual_

### Integrante 2: _nombre_

_Respuesta individual_

### Integrante 3: _nombre_

_Respuesta individual_

### Integrante 4: _nombre_

_Respuesta individual_

## Checklist de cierre — Semana 3

- [ ] Limpié el corpus y documenté cada decisión de limpieza.
- [ ] Analicé POS, NER y dependencias con spaCy usando nlp.pipe.
- [ ] Modelé temas con LDA y los comparé con BERTopic.
- [ ] Comparé Naive Bayes, SVM y una RNN con F1 macro en la misma partición.
- [ ] Repetí el experimento con 3, 5, 7, 9 y 11 clases contra la línea de azar.
- [ ] Repliqué la clasificación en Spark MLlib, consulté con Spark SQL y una función de ventana, e hice un conteo MapReduce con RDD.
- [ ] Comparé un LLM zero-shot y embeddings con mi mejor modelo, medí el drift con PSI y guardé resultados en Parquet.
- [ ] Completé la tabla de alternativas y la reflexión individual.

**Siguiente paso:** Esta semana completa los temas del módulo (datos tabulares, grafos, texto y procesamiento distribuido) y prepara el proyecto integrador con ArXiv: PySpark, spaCy, LDA vs. BERTopic y hoja de ruta MLOps.